# Suite PROP — Randomized properties (fast-check)

Invariants that must hold for all inputs, checked on generated natives and random graphs. fast-check shrinks any
failure to a minimal example. A fixed seed makes runs reproducible (Hypothesis in the Python suite).

In [ ]:
import fc from "fast-check";

import { JSON, Plain, Proxies, Reachable, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { DecodeError } from "@mbse/schemas/Framework/Errors";
import type { PlainMap } from "@mbse/schemas/Framework/Plain";
import { assert, raises, same_graph, text } from "./support.js";

const store = new Proxies.OfStore();

const CHECKS = { numRuns: 200, seed: 20260928 };

let Node = new S.OfObject.Builder().name("Node").ref().properties(text("id"), text("s"), text("i", BigInt), text("f", Number), text("b", Boolean),
  text("raw", Uint8Array)).create();
const Edge = new S.OfRelation.Builder().name("Edge").links("src", "dst").properties(text("label"), text("w", Number)).create();
Node = new S.OfObject.Builder(Node).relations((r) => r.name("out").of(Edge).me("src"), (r) => r.name("in").of(Edge).me("dst")).update();
store.register(Node);
store.register(Edge);
const B = store;
const validate = Validators.Validate(B);

const anyText = fc.string({ unit: fc.integer({ min: 0, max: 0xffff }).map((u) => String.fromCharCode(u)), maxLength: 40 }); // includes lone surrogates
const anyFloat = fc.double(); // includes NaN, ±Infinity, -0 and subnormals
const anyInt = fc.bigInt({ min: -(2n ** 200n), max: 2n ** 200n });
const natives = new Map<unknown, fc.Arbitrary<unknown>>([
  [String, anyText], [BigInt, anyInt], [Number, anyFloat], [Boolean, fc.boolean()], [Uint8Array, fc.uint8Array({ maxLength: 64 })],
]);
const tokens = [...natives.keys()];

function sameValue(a: unknown, b: unknown): boolean {
  if (typeof a === "number" && typeof b === "number") return Object.is(a, b) || (Number.isNaN(a) && Number.isNaN(b));
  if (a instanceof Uint8Array && b instanceof Uint8Array) return Buffer.from(a).equals(Buffer.from(b)) && Object.getPrototypeOf(b) === Uint8Array.prototype;
  return typeof a === typeof b && a === b;
}

## PROP-01 · Every native value round-trips through Plain, JSON and YAML, keeping its exact type

In [ ]:
fc.assert(fc.property(fc.constantFrom(...tokens).chain((t) => fc.tuple(fc.constant(t), natives.get(t) as fc.Arbitrary<unknown>)), ([native, value]) => {
  const schema = new S.OfNative.Data(native);
  assert(sameValue(Plain.FromPlain(B)(schema, Plain.ToPlain(store)(schema, value)), value));
  assert(sameValue(JSON.FromJSON(B)(schema, JSON.ToJSON(store)(schema, value)), value));
  assert(sameValue(YAML.FromYAML(B)(schema, YAML.ToYAML(store)(schema, value)), value));
}), CHECKS);

## PROP-02 · Lone surrogates survive both text encodings (YAML escapes them)

In [ ]:
for (const value of ["\ud800", "a\udfffb", "\udc00\ud800"]) {
  const schema = new S.OfNative.Data(String);
  assert(YAML.FromYAML(B)(schema, YAML.ToYAML(store)(schema, value)) === value);
  assert(JSON.FromJSON(B)(schema, JSON.ToJSON(store)(schema, value)) === value);
}
assert(YAML.ToYAML(store)(new S.OfNative.Data(String), "\ud800x") === '"\\uD800x"\n');

## PROP-03 · Mismatched native types are always rejected by `from_plain`

In [ ]:
fc.assert(fc.property(fc.constantFrom(...tokens), fc.constantFrom(...tokens).chain((t) => fc.tuple(fc.constant(t), natives.get(t) as fc.Arbitrary<unknown>)),
  (native, [source, value]) => { // as in Python: skip plain forms that legitimately match
    const schema = new S.OfNative.Data(native);
    const plain = new S.OfNative.Data(source).to_plain(value);
    if (S.isNativeOf(native, plain) || (native === Uint8Array && typeof plain === "string") || (native === Number && typeof plain === "string")) return;
    raises(DecodeError, () => schema.from_plain(plain));
  }), CHECKS);

## PROP-04 · Random graphs: round trips, validity, reachability and single-object snapshots

In [ ]:
const optional = <T>(arb: fc.Arbitrary<T>) => fc.option(arb, { nil: undefined });
const nodeProps = fc.record({ s: optional(anyText), i: optional(anyInt), f: optional(anyFloat), b: optional(fc.boolean()),
  raw: optional(fc.uint8Array({ maxLength: 16 })) });
const graphs = fc.integer({ min: 1, max: 6 }).chain((size) => fc.tuple(
  fc.array(nodeProps, { minLength: size, maxLength: size }),
  fc.array(fc.tuple(fc.integer({ min: 0, max: size - 1 }), fc.integer({ min: 0, max: size - 1 }), optional(anyText), optional(anyFloat)),
    { maxLength: 10 }),
));

function component(size: number, edges: [number, number, unknown, unknown][], start: number): Set<number> {
  const neighbours = new Map<number, Set<number>>(Array.from({ length: size }, (_, i) => [i, new Set<number>()]));
  for (const [src, dst] of edges) {
    neighbours.get(src)?.add(dst);
    neighbours.get(dst)?.add(src);
  }
  const seen = new Set([start]);
  const todo = [start];
  while (todo.length > 0) for (const n of neighbours.get(todo.pop() as number) ?? []) if (!seen.has(n)) { seen.add(n); todo.push(n); }
  return seen;
}

fc.assert(fc.property(graphs, ([props, edges]) => {
  const nodes = props.map((p, index) => {
    const builder = B.Node().id(String(index));
    for (const [name, value] of Object.entries(p)) if (value !== undefined) builder[name](value);
    return builder.create();
  });
  for (const [src, dst, label, weight] of edges) {
    B.Node(nodes[src]).out((x: any) => { x.dst(nodes[dst]); if (label !== undefined) x.label(label); if (weight !== undefined) x.w(weight); return x; }).update();
  }
  const root = nodes[0];
  const snapshot = Plain.ToPlain(store).Reachable(Node, root);

  assert(validate.Reachable(Node, root).length === 0);
  const reached = Reachable.of(root);
  const expected = new Set([...component(nodes.length, edges, 0)].map((i) => nodes[i].identity()));
  assert(reached[0] === root && reached.length === expected.size && reached.every((o) => expected.has(o.identity())));
  assert((snapshot.get("objects") as PlainMap).size === reached.length);
  assert((Plain.ToPlain(store)(Node, root).get("objects") as PlainMap).size === 1);

  const restored = Plain.FromPlain(B).Reachable(Node, snapshot) as Proxies.Instance;
  assert(same_graph(Plain.ToPlain(store).Reachable(Node, restored), snapshot));
  const jsonText = JSON.ToJSON(store).Reachable(Node, root);
  assert(same_graph(JSON.loads(jsonText), snapshot));
  assert(same_graph(Plain.ToPlain(store).Reachable(Node, JSON.FromJSON(B).Reachable(Node, jsonText) as Proxies.Instance), snapshot));
  const yamlText = YAML.ToYAML(store).Reachable(Node, root);
  assert(same_graph(YAML.loads(yamlText), snapshot));
  assert(same_graph(Plain.ToPlain(store).Reachable(Node, YAML.FromYAML(B).Reachable(Node, yamlText) as Proxies.Instance), snapshot));
}), { ...CHECKS, numRuns: 200 });

## PROP-05 · Entries form a set: the count equals the number of distinct entries under schema equality

In [ ]:
const keyOf = (value: unknown) => (typeof value === "number" ? (Number.isNaN(value) ? "float:nan" : `float:${Object.is(value, -0) ? "-0" : value}`) : `str:${value}`);
fc.assert(fc.property(fc.array(fc.tuple(optional(fc.constantFrom("a", "b")), optional(fc.constantFrom(0.0, -0.0, NaN, 1.0))), { maxLength: 12 }), (props) => {
  const owner = B.Node().id("o").create();
  const target = B.Node().id("t").create();
  for (const [label, weight] of props) {
    B.Node(owner).out((x: any) => { x.dst(target); if (label !== undefined) x.label(label); if (weight !== undefined) x.w(weight); return x; }).update();
  }
  const distinct = new Set(props.map(([l, w]) => `${l === undefined ? "-" : keyOf(l)}|${w === undefined ? "-" : keyOf(w)}`));
  const written = ((Plain.ToPlain(store)(Node, owner).get("objects") as PlainMap).get("s0") as PlainMap).get("out") as unknown[] | undefined;
  assert((written?.length ?? 0) === distinct.size);
}), CHECKS);